## Let the Neurons Die Part II - Poisoned Data Construction with the idea of Inverting Gradients Attack

This notebook is an example for reconstructing input training data in a malicious way using weight and gradient information. The reconstructed datapoint(s) will be used for victim model training and the expected outcome should be seeing dead neurons and model never being converged. The model is a custom model with **ReLu** activation function.

### Startup

The startup code should be the same as the *Inverting Gradients Attack*.
Please note that in order to be 100% that we are running the modified `breaching` framework, remember to uninstall the `breaching` library if you installed the package before via `pip3 uninstall breaching`.

In [ ]:
try:
    import breaching
except ModuleNotFoundError:
    # You only really need this safety net if you want to run these notebooks directly in the examples directory
    # Don't worry about this if you installed the package or moved the notebook to the main directory.
    import os; os.chdir("..")
    import breaching
    
import torch
import numpy as np
import os
%load_ext autoreload
%autoreload 2

# Redirects logs directly into the jupyter notebook
import logging, sys
logging.basicConfig(level=logging.INFO, handlers=[logging.StreamHandler(sys.stdout)], format='%(message)s')
logger = logging.getLogger()

In [ ]:
# Parameters
cases = ["11_single_mnist", "12_batch_mnist", "1_single_image_small", "6_large_batch_cifar"]
nets = ['mnistnet', "convnet"]
frontend_attacks = ["data_ordering", "soft_knockout"]

# Other parameters should be set elsewhere:
# * batch number per run needs to be specified in *.yaml files 
#   under inverting_gradients/breaching/breaching/config/case/data
# * Frontend attack (whether knockout or data ordering) should be 
#   specified in inverting_gradients/breaching/breaching/cases/models/model_preparation.py
# * Other customized models should also be added in model_preparation.py

case_name = cases[0] # for logging purpose only
case = "case=" + case_name
net = nets[0]
num_classes = 10 # Should be set based on dataset. e.g., MNIST has 10 classes, CIFAR 100 has 100 classes
frontend_attack = frontend_attacks[0] # for logging purpose only
num_batch = 1 # for logging purpose only. Data got from target .yaml file

# for data storing purpose
store_path = os.getcwd() + "/from_" + frontend_attack + "_" + str(num_batch)
try: 
    os.mkdir(store_path) 
except OSError as error: 
    print(error)  

## Poisoned data creation

In [ ]:
# Uncomment if create one single poisoned data point
cfg = breaching.get_config(overrides=[case, "attack=invertinggradients"])
          
device = torch.device(f'cuda') if torch.cuda.is_available() else torch.device('cpu')
torch.backends.cudnn.benchmark = cfg.case.impl.benchmark
setup = dict(device=device, dtype=getattr(torch, cfg.case.impl.dtype))
setup

cfg.case.data.partition="unique-class"
cfg.case.user.user_idx = 0
cfg.case.model=net

for id in range(num_classes):
    cfg.case.user.user_idx = id
    user, server, model, loss_fn = breaching.cases.construct_case(cfg.case, setup)
    attacker = breaching.attacks.prepare_attack(server.model, server.loss, cfg.attack, setup)
    breaching.utils.overview(server, user, attacker)

    server_payload = server.distribute_payload()
    shared_data, true_user_data = user.compute_local_updates(server_payload)

    print(true_user_data['data'].shape)
    user.plot(true_user_data)

    reconstructed_user_data, stats = attacker.reconstruct([server_payload], [shared_data], {}, dryrun=cfg.dryrun)

    user.plot(reconstructed_user_data)
    
    torch.save(reconstructed_user_data, os.path.join(store_path, 'class_'+str(id)+'_reconstructed_user_data_'+ case_name + '.pt'))
    np.save(os.path.join(store_path, 'class_'+str(id)+'_reconstructed_user_data_'+ case_name + '.npy'), reconstructed_user_data['data'].cpu().detach().numpy())
